In [ ]:
# Step 1 (about 1 min): pick all five ride zips together when the button appears
import zipfile, io, json, time, requests, numpy as np, pandas as pd
from google.colab import files
up = files.upload()


In [ ]:
# Step 2 (about 10 s): find spots that were hit hard on 2 or more rides
MIN_PEAK = 8      # m/s2, sharp hits only
CLUSTER_M = 40    # hits closer than this count as one spot
rows = []
for name, data in up.items():
    z = zipfile.ZipFile(io.BytesIO(data))
    ride = name.split('RoadCheck_')[-1][:30]
    for n in z.namelist():
        if n.endswith('_bumps.geojson'):
            for f in json.loads(z.read(n))['features']:
                x, y = f['geometry']['coordinates']
                rows.append(dict(ride=ride, lat=y, lon=x, peak=f['properties']['peak_v'], kmh=f['properties']['speed_kmh']))
b = pd.DataFrame(rows)
b = b[b.peak >= MIN_PEAK].reset_index(drop=True)
m = np.cos(np.radians(b.lat.mean())) * 111320
k = 110574
b['c'] = -1
c = 0
for i in range(len(b)):
    if b.c[i] >= 0:
        continue
    d = np.hypot((b.lat - b.lat[i]) * k, (b.lon - b.lon[i]) * m)
    b.loc[(d < CLUSTER_M) & (b.c < 0), 'c'] = c
    c += 1
spots = (b.groupby('c').agg(lat=('lat', 'mean'), lon=('lon', 'mean'), rides=('ride', 'nunique'),
         hits=('ride', 'size'), peak=('peak', 'max'), kmh=('kmh', 'median')).reset_index(drop=True))
spots = spots[spots.rides >= 2].sort_values('peak', ascending=False).reset_index(drop=True)
print(len(b), 'sharp hits from', b.ride.nunique(), 'rides ->', len(spots), 'spots seen on 2+ rides')


In [ ]:
# Step 3 (about 1-2 min): ask OpenStreetMap what is at each spot
UA = {'User-Agent': 'RoadCheck-analysis (somdeepkundu@gmail.com)'}
NEAR_M = 30   # a feature this close can explain the bump
pad = 0.003
S, W, N, E = b.lat.min()-pad, b.lon.min()-pad, b.lat.max()+pad, b.lon.max()+pad
bb = f"({S},{W},{N},{E})"
q = ('[out:json][timeout:60];('
     f'node["traffic_calming"]{bb};node["railway"="level_crossing"]{bb};'
     f'node["highway"~"crossing|traffic_signals"]{bb};way["bridge"]["highway"]{bb};'
     ');out center tags;')
feats = []
for url in ['https://overpass-api.de/api/interpreter', 'https://overpass.private.coffee/api/interpreter']:
    try:
        r = requests.post(url, data={'data': q}, headers=UA, timeout=90)
        feats = r.json()['elements']
        print('Overpass ok:', len(feats), 'features from', url)
        break
    except Exception as e:
        print('Overpass failed at', url, type(e).__name__)
def label(t):
    if 'traffic_calming' in t: return 'speed breaker (' + t['traffic_calming'] + ')'
    if t.get('railway') == 'level_crossing': return 'level crossing'
    if t.get('highway') in ('crossing', 'traffic_signals'): return t['highway'].replace('_', ' ')
    if 'bridge' in t: return 'bridge'
    return 'other'
pts = []
for f in feats:
    la = f.get('lat') or f.get('center', {}).get('lat')
    lo = f.get('lon') or f.get('center', {}).get('lon')
    if la: pts.append((la, lo, label(f.get('tags', {}))))
near, dist, road = [], [], []
for s in spots.itertuples():
    best = (None, 1e9)
    for la, lo, lab in pts:
        d = np.hypot((la - s.lat) * k, (lo - s.lon) * m)
        if d < best[1]: best = (lab, d)
    near.append('OSM not loaded' if not pts else (best[0] if best[1] <= NEAR_M else 'none within %d m' % NEAR_M))
    dist.append(round(best[1]) if best[1] <= NEAR_M else None)
    try:
        j = requests.get('https://nominatim.openstreetmap.org/reverse', headers=UA, timeout=30,
                         params=dict(format='jsonv2', lat=s.lat, lon=s.lon, zoom=17)).json()
        a = j.get('address', {})
        road.append(a.get('road') or a.get('suburb') or a.get('village') or j.get('display_name', '')[:40])
    except Exception:
        road.append('lookup failed')
    time.sleep(1.1)   # Nominatim allows 1 request per second
spots['near_feature'], spots['dist_m'], spots['road_or_place'] = near, dist, road
spots = spots.round({'lat': 5, 'lon': 5, 'kmh': 0})
display(spots)
spots.to_csv('spots_explained.csv', index=False)
files.download('spots_explained.csv')
